# AI Arbitrage Model

This notebook builds a simple AI-assisted arbitrage detection model for cryptocurrency markets. The goal is not to guarantee profit, but to show a clean data science workflow for identifying possible price discrepancies across exchanges.

The notebook includes:

1. Simulated multi-exchange crypto price data
2. Arbitrage spread calculation
3. Transaction-cost adjustment
4. Feature engineering
5. A machine learning model to classify potentially profitable opportunities
6. Backtest-style evaluation

**Important:** This is an educational project. Real arbitrage requires live exchange APIs, order-book depth, latency control, fees, slippage, transfer constraints, and execution-risk management.


## 1. Import Libraries

We begin by importing the main Python libraries used in the project.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

np.random.seed(42)

## 2. Create Simulated Exchange Price Data

In a real project, this section would be replaced with API calls from exchanges such as Binance, Coinbase, Kraken, or Bybit. For a portfolio project, simulated data is acceptable because it allows the model structure to be shown clearly.

We simulate BTC prices across three exchanges. Each exchange price follows the same general market movement, but with small exchange-specific deviations.

In [ ]:
n = 3000
base_price = 60000 + np.cumsum(np.random.normal(0, 25, n))

df = pd.DataFrame({
    "timestamp": pd.date_range(start="2025-01-01", periods=n, freq="min"),
    "binance_price": base_price + np.random.normal(0, 12, n),
    "coinbase_price": base_price + np.random.normal(0, 14, n),
    "kraken_price": base_price + np.random.normal(0, 13, n),
})

df.head()

## 3. Calculate Arbitrage Spreads

A simple arbitrage opportunity exists when the same asset trades at different prices across exchanges. The basic idea is:

- Buy on the cheaper exchange
- Sell on the more expensive exchange
- Keep the spread after transaction costs

This simplified version assumes the asset can be bought and sold instantly. Real execution is more difficult because of latency, fees, liquidity, and order-book depth.

In [ ]:
price_cols = ["binance_price", "coinbase_price", "kraken_price"]

df["lowest_price"] = df[price_cols].min(axis=1)
df["highest_price"] = df[price_cols].max(axis=1)
df["buy_exchange"] = df[price_cols].idxmin(axis=1).str.replace("_price", "")
df["sell_exchange"] = df[price_cols].idxmax(axis=1).str.replace("_price", "")

df["raw_spread"] = df["highest_price"] - df["lowest_price"]
df["raw_spread_pct"] = df["raw_spread"] / df["lowest_price"]

df[["timestamp", "buy_exchange", "sell_exchange", "lowest_price", "highest_price", "raw_spread_pct"]].head()

## 4. Adjust for Trading Costs

A spread is not enough. The opportunity must remain profitable after costs. We include a simple cost assumption:

- Buy fee: 0.10%
- Sell fee: 0.10%
- Slippage buffer: 0.05%

Total cost = 0.25%.

In [ ]:
buy_fee = 0.001
sell_fee = 0.001
slippage_buffer = 0.0005

total_cost = buy_fee + sell_fee + slippage_buffer

df["net_spread_pct"] = df["raw_spread_pct"] - total_cost
df["profitable_now"] = (df["net_spread_pct"] > 0).astype(int)

df[["raw_spread_pct", "net_spread_pct", "profitable_now"]].head()

## 5. Feature Engineering

The AI component should not simply look at the current spread. It should also learn from recent market behavior. We create features based on:

- Recent volatility
- Recent average spread
- Spread change
- Price momentum

The target variable asks whether the arbitrage opportunity remains profitable shortly after detection. This is more realistic than only asking whether the current spread is positive, because execution takes time.

In [ ]:
df["mid_price"] = df[price_cols].mean(axis=1)
df["price_return"] = df["mid_price"].pct_change()
df["spread_change"] = df["raw_spread_pct"].diff()

df["volatility_10"] = df["price_return"].rolling(10).std()
df["volatility_30"] = df["price_return"].rolling(30).std()
df["avg_spread_10"] = df["raw_spread_pct"].rolling(10).mean()
df["avg_spread_30"] = df["raw_spread_pct"].rolling(30).mean()
df["momentum_10"] = df["mid_price"].pct_change(10)
df["momentum_30"] = df["mid_price"].pct_change(30)

# Future profitability after a short execution delay
execution_delay = 3
df["future_net_spread_pct"] = df["net_spread_pct"].shift(-execution_delay)
df["target"] = (df["future_net_spread_pct"] > 0).astype(int)

df = df.dropna().reset_index(drop=True)

df.head()

## 6. Prepare Training and Testing Data

The model will try to classify whether a detected opportunity is still profitable after a short delay.

In [ ]:
features = [
    "raw_spread_pct",
    "net_spread_pct",
    "spread_change",
    "volatility_10",
    "volatility_30",
    "avg_spread_10",
    "avg_spread_30",
    "momentum_10",
    "momentum_30",
]

X = df[features]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=False
)

X_train.shape, X_test.shape

## 7. Train the AI Model

We use a Random Forest classifier because it can capture nonlinear relationships and interactions among spread, volatility, and momentum features.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=10,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, predictions))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions))

## 8. Feature Importance

Feature importance helps explain which variables the model used most heavily.

In [ ]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

importance

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(importance["feature"], importance["importance"])
plt.gca().invert_yaxis()
plt.title("AI Arbitrage Model Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## 9. Backtest-Style Signal Evaluation

The model gives a probability that an arbitrage opportunity will remain profitable after the execution delay. We only act when the model is confident enough.

This section uses a confidence threshold. Higher thresholds usually mean fewer trades but potentially better average quality.

In [ ]:
test_results = df.iloc[X_test.index].copy()
test_results["model_probability"] = probabilities

threshold = 0.60
test_results["trade_signal"] = (test_results["model_probability"] >= threshold).astype(int)

test_results["strategy_return"] = test_results["trade_signal"] * test_results["future_net_spread_pct"]
test_results["cumulative_strategy_return"] = (1 + test_results["strategy_return"]).cumprod() - 1

num_trades = test_results["trade_signal"].sum()
avg_trade_return = test_results.loc[test_results["trade_signal"] == 1, "strategy_return"].mean()
win_rate = (test_results.loc[test_results["trade_signal"] == 1, "strategy_return"] > 0).mean()
total_return = test_results["cumulative_strategy_return"].iloc[-1]

summary = pd.DataFrame({
    "Metric": ["Number of Trades", "Average Trade Return", "Win Rate", "Total Strategy Return"],
    "Value": [num_trades, avg_trade_return, win_rate, total_return]
})

summary

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(test_results["timestamp"], test_results["cumulative_strategy_return"])
plt.title("Cumulative Strategy Return: AI Arbitrage Signal")
plt.xlabel("Time")
plt.ylabel("Cumulative Return")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 10. Interpretation

This notebook shows how an AI-assisted arbitrage model can be structured. The basic arbitrage logic identifies price differences across exchanges. The machine learning layer then estimates whether those opportunities are likely to remain profitable after a short delay.

The most important lesson is that arbitrage is not only about finding a spread. A real system must also ask whether the spread is large enough after costs and whether it can survive long enough to be executed.

A stronger real-world version would use:

- Live exchange API data
- Bid/ask prices instead of last traded prices
- Order-book depth
- Exchange-specific fees
- Slippage estimates
- Latency measurement
- Real execution logs
- Risk controls

For a portfolio project, this notebook demonstrates the core idea clearly: use data science and machine learning to convert an arbitrage intuition into a systematic, testable trading framework.